# 01 - Load & QC

Veri yukleme, yapisal butunluk, zaman/sinyal dogrulama, QC bayraklari.

Veri Google Drive klasorunden cekilir. Klasor link ile herkese acik oldugu
icin kimlik dogrulama yok: API anahtari, OAuth, client_secrets.json gerekmez.

In [ ]:
%pip install -q pyyaml pandas numpy pyarrow gdown

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ANALYSIS_ROOT = Path.cwd()
if not (ANALYSIS_ROOT / "config.yaml").exists():
    ANALYSIS_ROOT = ANALYSIS_ROOT.parent
sys.path.insert(0, str(ANALYSIS_ROOT))

from src.drive_sync import sync_data
from src.loader import load_all
from src.qc import (
    check_structural_integrity,
    check_timing,
    check_signals,
    flag_trials,
    add_analysis_mask,
    check_format_regression,
)

with open(ANALYSIS_ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

RAW_DIR = ANALYSIS_ROOT / config["paths"]["raw_dir"]
INTERIM_DIR = ANALYSIS_ROOT / config["paths"]["interim_dir"]
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
print(f"Kok: {ANALYSIS_ROOT}")

## 0. Drive'dan veri cek

In [ ]:
# Var olan dosyalar tekrar indirilmez. Yeni katilimci geldiginde
# bu hucreyi tekrar calistirmak yeterli.
sync_data(config["drive"]["folder_id"], RAW_DIR);

## 1. Kesif ve yukleme

In [ ]:
df_samples, df_trials, metadata, report = load_all(RAW_DIR)

print(f"Oturumlar: {len(report)}")
n_sel = sum(1 for s in report if s["status"] == "selected")
n_inc = sum(1 for s in report if s["status"] == "incomplete")
print(f"  Secilen:   {n_sel}")
print(f"  Yarim:     {n_inc}")
print(f"Sample:      {len(df_samples):,}")
print(f"Trial:       {len(df_trials)}")
print(f"Metadata:    {len(metadata)} dosya")

if report:
    cols = [
        "participant_id", "session_id", "status",
        "has_metadata", "has_timeseries", "has_trial_summary",
        "measurement_trial_count", "warnings",
    ]
    display(pd.DataFrame(report)[cols])

## 2. Yapisal butunluk

In [ ]:
issues = check_structural_integrity(df_trials, metadata, config)

if issues:
    df_iss = pd.DataFrame(issues)
    n_fail = int((df_iss["status"] == "FAIL").sum())
    n_warn = len(df_iss) - n_fail
    print(f"{len(df_iss)} sorun  (FAIL: {n_fail}, WARN: {n_warn})")
    display(df_iss.sort_values("status"))
else:
    print("Yapisal sorun yok.")

## 3. Zaman ve ornekleme

In [ ]:
timing = check_timing(df_samples, config)

if timing.empty:
    print("Veri yok, atlaniyor.")
else:
    problem_mask = (
        timing["has_time_reversal"]
        | timing["has_gap"]
        | timing["has_dup_index"]
        | timing["has_skip_index"]
        | (timing["nan_count"] > 0)
        | (timing["angle_out_of_range"] > 0)
    )
    problems = timing[problem_mask]
    if len(problems) > 0:
        n_prob = len(problems)
        print(f"{n_prob} trial'da zaman/ornekleme sorunu:")
        display(problems)
    else:
        print("Zaman ve ornekleme sorunsuz.")

    dt_avg = timing["dt_mean"].mean()
    dt_std_avg = timing["dt_std"].mean()
    dt_max = timing["dt_max_dev"].max()
    print()
    print(f"dt ort: {dt_avg:.6f} s")
    print(f"dt std ort: {dt_std_avg:.8f} s")
    print(f"dt max sapma: {dt_max:.6f} s")

## 4. Sinyal akil sagligi

In [ ]:
signals = check_signals(df_samples, config)

if signals.empty:
    print("Veri yok, atlaniyor.")
else:
    warn_thr = config["qc"]["velocity_correlation_warn"]

    low_corr = signals[
        (signals["cart_vel_corr"] < warn_thr)
        | (signals["omega_corr"] < warn_thr)
    ]
    if len(low_corr) > 0:
        print(f"{len(low_corr)} trial'da hiz-pozisyon korelasyonu < {warn_thr}")
        display(low_corr[["participant_id", "trial_id", "n_segments", "cart_vel_corr", "omega_corr"]])
    else:
        print(f"Hiz-pozisyon korelasyonu tum triallarda >= {warn_thr}")

    force_bad = signals[~signals["force_ok"]]
    if len(force_bad) > 0:
        print()
        print(f"{len(force_bad)} trial'da force tutarsiz:")
        display(force_bad[["participant_id", "trial_id"]])
    else:
        max_f = config["physics"]["max_force_n"]
        print()
        print(f"Force tutarliligi OK (input_applied x {max_f} N)")

    phase_bad = signals[~signals["phase_reset_ok"]]
    if len(phase_bad) > 0:
        print()
        print(f"{len(phase_bad)} trial'da phase/is_resetting tutarsiz:")
        display(phase_bad[["participant_id", "trial_id"]])
    else:
        print()
        print("phase ve is_resetting tutarli.")

## 5. Trial gecerliligi ve sample maskesi

In [ ]:
df_trials = flag_trials(df_trials, df_samples, config)
df_samples = add_analysis_mask(df_samples, df_trials)

if df_trials.empty:
    print("Veri yok, atlaniyor.")
else:
    n_pass = int(df_trials["qc_pass"].sum())
    n_fail = len(df_trials) - n_pass
    print(f"Trial: {len(df_trials)}  (pass: {n_pass}, fail: {n_fail})")

    if n_fail > 0:
        print()
        print("Dusen triallar:")
        fail_cols = ["participant_id", "trial_id", "noise_level_id", "practice", "qc_flags"]
        display(df_trials[~df_trials["qc_pass"]][fail_cols])

    n_inc = int(df_samples["analysis_include"].sum())
    n_exc = len(df_samples) - n_inc
    print()
    print(f"Sample: {len(df_samples):,}  (include: {n_inc:,}, exclude: {n_exc:,})")

    meas = df_trials[(df_trials["practice"] == 0) & df_trials["qc_pass"]]
    print()
    print(f"Analize giren measurement trial: {len(meas)}")
    for pid in sorted(meas["participant_id"].unique()):
        n = int((meas["participant_id"] == pid).sum())
        print(f"  {pid}: {n}")

## 6. Format regresyon takibi

In [ ]:
regression = check_format_regression(metadata, config)

if regression:
    df_reg = pd.DataFrame(regression)
    present = int(df_reg["present"].sum())
    missing = len(df_reg) - present
    print(f"Istenen alanlar: {len(df_reg)}  (mevcut: {present}, eksik: {missing})")
    if missing > 0:
        print()
        print("Eksik alanlar:")
        display(df_reg[~df_reg["present"]])
else:
    print("Metadata yok, kontrol yapilamadi.")

## Cikti

In [ ]:
if df_samples.empty or df_trials.empty:
    print("Veri yok, cikti yazilmiyor.")
else:
    df_samples.to_parquet(INTERIM_DIR / "samples_clean.parquet", index=False)
    df_trials.to_parquet(INTERIM_DIR / "trials_clean.parquet", index=False)
    print(f"samples_clean.parquet  ({len(df_samples):,} satir)")
    print(f"trials_clean.parquet   ({len(df_trials)} satir)")